In [1]:
from __future__ import annotations

"""
exploration.py — Analyse exploratoire complète du dataset Retail Customers.

Sections :
  1) Structure globale
  2) Qualité des données (valeurs manquantes, doublons, colonnes constantes,
     cardinalité, outliers IQR)
  3) Contrôles métier (Age, Satisfaction, SupportTickets, RegistrationDate)
  4) Corrélation & multicolinéarité
  5) Distribution de la cible Churn
  6) ACP (PCA) — réduction dimensionnelle + visualisation 2D
  7) Feature Engineering — nouvelles variables dérivées
  8) Export des rapports CSV + figures PNG
"""

from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")  # backend non-interactif (compatible sans écran)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# ============================================================
# Chemins — compatibles script ET notebook Jupyter
# ============================================================
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"notebooks", "src"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH    = PROJECT_ROOT / "data" / "raw" / "retail_customers_COMPLETE_CATEGORICAL.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_PATH     :", RAW_PATH)
print("RAW exists?  :", RAW_PATH.exists())


# ============================================================
# Fonctions utilitaires locales
# (redondantes avec utils.py mais autonomes pour le notebook)
# ============================================================

def safe_to_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def safe_to_datetime(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", dayfirst=True)


def churn_to_binary(s: pd.Series) -> pd.Series:
    """Convertit Churn en 0/1, robuste aux formats texte/booléen/numérique."""
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float).round().clip(0, 1).astype(int)

    mapping = {
        "1": 1, "true": 1, "yes": 1, "y": 1, "oui": 1, "parti": 1, "churn": 1,
        "0": 0, "false": 0, "no": 0, "n": 0, "non": 0, "fidèle": 0, "loyal": 0,
    }
    x = s.astype(str).str.strip().str.lower().map(mapping)
    x = x.fillna(pd.to_numeric(s, errors="coerce"))
    return x.fillna(0).astype(float).round().clip(0, 1).astype(int)


def iqr_outlier_rate(series: pd.Series) -> float:
    """Retourne le % de valeurs hors moustaches IQR."""
    x = safe_to_numeric(series).dropna()
    if x.empty:
        return 0.0
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    low  = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    return float(((x < low) | (x > high)).mean() * 100)


def section(title: str) -> None:
    """Affiche un séparateur de section lisible."""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")


# ============================================================
# 1) CHARGEMENT & STRUCTURE GLOBALE
# ============================================================

def analyse_structure(df: pd.DataFrame) -> tuple[list, list]:
    """Affiche la structure globale du DataFrame."""
    section("1) STRUCTURE GLOBALE")

    print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    print(f"\nPremières lignes :")
    print(df.head(3).to_string())

    print(f"\nTypes de colonnes :")
    print(df.dtypes.value_counts().to_string())

    num_cols = df.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = [c for c in df.columns if c not in num_cols]

    print(f"\nColonnes numériques ({len(num_cols)}) :")
    print("  " + ", ".join(num_cols[:15]) + (" ..." if len(num_cols) > 15 else ""))
    print(f"\nColonnes catégorielles / texte ({len(cat_cols)}) :")
    print("  " + ", ".join(cat_cols[:15]) + (" ..." if len(cat_cols) > 15 else ""))

    print(f"\nStatistiques descriptives (numériques) :")
    print(df[num_cols].describe().round(2).to_string())

    return num_cols, cat_cols


# ============================================================
# 2) QUALITÉ DES DONNÉES
# ============================================================

def analyse_qualite(df: pd.DataFrame, num_cols: list, cat_cols: list) -> pd.DataFrame:
    """
    Analyse complète de la qualité :
    - Valeurs manquantes
    - Lignes dupliquées
    - Colonnes constantes
    - Cardinalité des catégorielles
    - Outliers IQR
    """
    section("2) QUALITÉ DES DONNÉES")

    # --- Valeurs manquantes ---
    print("\n── Valeurs manquantes ──")
    missing = (df.isna().sum()).sort_values(ascending=False)
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    missing_df = pd.DataFrame({"count": missing, "pct": missing_pct.round(2)})
    missing_nonzero = missing_df[missing_df["count"] > 0]
    if missing_nonzero.empty:
        print("  ✅ Aucune valeur manquante.")
    else:
        print(missing_nonzero.head(15).to_string())

    # --- Doublons ---
    print("\n── Lignes dupliquées ──")
    dup = df.duplicated().sum()
    print(f"  Doublons exacts : {dup} ({dup / len(df) * 100:.2f}%)")
    if dup > 0:
        print("  ⚠️  Des doublons ont été détectés — à supprimer avant modélisation.")

    # --- Colonnes constantes ---
    print("\n── Colonnes constantes (variance nulle) ──")
    constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    if constant_cols:
        print(f"  ⚠️  Colonnes à supprimer : {constant_cols}")
    else:
        print("  ✅ Aucune colonne constante.")

    # --- Cardinalité des catégorielles ---
    print("\n── Cardinalité des colonnes catégorielles ──")
    cardinality = {}
    for c in cat_cols:
        card = df[c].nunique(dropna=True)
        cardinality[c] = card
    card_df = pd.DataFrame.from_dict(cardinality, orient="index", columns=["n_unique"])
    card_df = card_df.sort_values("n_unique", ascending=False)
    print(card_df.to_string())

    high_card = card_df[card_df["n_unique"] >= 20]
    if not high_card.empty:
        print(f"\n  ⚠️  Colonnes à forte cardinalité (>=20) : {high_card.index.tolist()}")
        print("     → Envisager : Target Encoding, regroupement ou suppression.")

    # --- Outliers IQR ---
    print("\n── Taux d'outliers IQR (colonnes numériques) ──")
    outlier_stats = [(c, iqr_outlier_rate(df[c])) for c in num_cols]
    outlier_df = (
        pd.DataFrame(outlier_stats, columns=["column", "outlier_pct"])
        .sort_values("outlier_pct", ascending=False)
        .reset_index(drop=True)
    )
    print(outlier_df.head(12).to_string(index=False))

    high_outlier = outlier_df[outlier_df["outlier_pct"] > 5]
    if not high_outlier.empty:
        print(f"\n  ⚠️  Colonnes avec >5% d'outliers : {high_outlier['column'].tolist()}")
        print("     → Envisager : clipping IQR, log-transformation ou imputation robuste.")

    return outlier_df


# ============================================================
# 3) CONTRÔLES MÉTIER
# ============================================================

def controles_metier(df: pd.DataFrame) -> pd.DataFrame:
    """
    Vérifie les contraintes métier connues des features :
    - Age          : intervalle attendu [15, 100]
    - Satisfaction : intervalle attendu [1, 5]
    - SupportTickets : doit être >= 0
    - RegistrationDate : doit être parseable
    """
    section("3) CONTRÔLES MÉTIER")

    domain_rows = []

    # Age
    if "Age" in df.columns:
        age = safe_to_numeric(df["Age"])
        bad = ((age < 15) | (age > 100)).mean() * 100
        domain_rows.append(("Age hors [15, 100]", round(bad, 2)))
        print(f"  Age        — valeurs hors [15,100] : {bad:.2f}%")
        print(f"    min={age.min():.0f}  max={age.max():.0f}  "
              f"NaN={age.isna().sum()} ({age.isna().mean()*100:.1f}%)")

    # Satisfaction
    if "Satisfaction" in df.columns:
        sat = safe_to_numeric(df["Satisfaction"])
        bad = ((sat < 1) | (sat > 5)).mean() * 100
        domain_rows.append(("Satisfaction hors [1, 5]", round(bad, 2)))
        print(f"\n  Satisfaction — valeurs hors [1,5] : {bad:.2f}%")
        print(f"    min={sat.min():.0f}  max={sat.max():.0f}  "
              f"valeurs uniques={sorted(sat.dropna().unique().tolist())[:10]}")

    # SupportTickets
    if "SupportTickets" in df.columns:
        st = safe_to_numeric(df["SupportTickets"])
        bad = (st < 0).mean() * 100
        domain_rows.append(("SupportTickets < 0", round(bad, 2)))
        print(f"\n  SupportTickets — valeurs négatives : {bad:.2f}%")
        print(f"    min={st.min():.0f}  max={st.max():.0f}  "
              f"valeurs aberrantes (999, -1) : {((st == 999) | (st == -1)).sum()}")

    # RegistrationDate
    date_candidates = [
        c for c in df.columns
        if c.lower() in {"registdate", "registrationdate", "registerdate", "signupdate"}
    ]
    if date_candidates:
        dcol = date_candidates[0]
        parsed = safe_to_datetime(df[dcol])
        bad = parsed.isna().mean() * 100
        domain_rows.append((f"{dcol} non parseable", round(bad, 2)))
        print(f"\n  {dcol} — taux non parseable : {bad:.2f}%")
        print(f"    Exemples bruts : {df[dcol].dropna().astype(str).head(5).tolist()}")
        print(f"    Plage parsée   : {parsed.min()} → {parsed.max()}")

    if domain_rows:
        domain_df = pd.DataFrame(domain_rows, columns=["check", "bad_pct"]).sort_values(
            "bad_pct", ascending=False
        )
        print(f"\n  Résumé contrôles métier :")
        print(domain_df.to_string(index=False))
    else:
        domain_df = pd.DataFrame(columns=["check", "bad_pct"])
        print("  Aucun contrôle métier applicable (colonnes absentes).")

    return domain_df


# ============================================================
# 4) CORRÉLATION & MULTICOLINÉARITÉ
# ============================================================

def analyse_correlation(df: pd.DataFrame, num_cols: list, figures_dir: Path) -> pd.DataFrame:
    """
    Calcule et affiche :
    - Heatmap de corrélation (sauvegardée en PNG)
    - Paires fortement corrélées (|r| > 0.8)
    - Recommandations de suppression
    """
    section("4) CORRÉLATION & MULTICOLINÉARITÉ")

    num_df = df[num_cols].copy()
    corr = num_df.corr(method="pearson")

    # --- Heatmap ---
    fig, ax = plt.subplots(figsize=(16, 12))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr,
        mask=mask,
        annot=len(num_cols) <= 20,
        fmt=".2f",
        cmap="RdYlGn",
        center=0,
        linewidths=0.3,
        ax=ax,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title("Matrice de corrélation — Features numériques", fontsize=14, pad=12)
    plt.tight_layout()
    heatmap_path = figures_dir / "correlation_heatmap.png"
    fig.savefig(heatmap_path, dpi=150)
    plt.close(fig)
    print(f"  Heatmap sauvegardée : {heatmap_path}")

    # --- Paires fortement corrélées ---
    THRESHOLD = 0.8
    upper = corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))
    pairs = (
        upper.stack()
        .reset_index()
        .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "correlation"})
    )
    high_corr = pairs[pairs["correlation"].abs() >= THRESHOLD].sort_values(
        "correlation", key=abs, ascending=False
    )

    print(f"\n  Paires avec |corrélation| ≥ {THRESHOLD} :")
    if high_corr.empty:
        print("  ✅ Aucune paire fortement corrélée.")
    else:
        print(high_corr.to_string(index=False))
        print(f"\n  ⚠️  {len(high_corr)} paire(s) détectée(s).")
        print("     → Envisager de conserver la feature la plus interprétable de chaque paire.")
        print("     → Ou utiliser l'ACP pour réduire la redondance.")

    return high_corr


# ============================================================
# 5) DISTRIBUTION DE LA CIBLE CHURN
# ============================================================

def analyse_churn(df: pd.DataFrame, figures_dir: Path) -> pd.Series:
    """
    Analyse la distribution de la variable cible Churn :
    - Distribution numérique + ratio déséquilibre
    - Graphique barplot + camembert
    - Taux de churn par segment (si RFMSegment / CustomerType présent)
    """
    section("5) CIBLE CHURN")

    if "Churn" not in df.columns:
        print("  ⚠️  Colonne 'Churn' absente du dataset.")
        return pd.Series(dtype=int)

    y = churn_to_binary(df["Churn"])

    # Distribution
    counts = y.value_counts().sort_index()
    pct    = y.value_counts(normalize=True).sort_index() * 100
    print("  Distribution Churn :")
    print(f"    Non-Churn (0) : {counts.get(0, 0):,}  ({pct.get(0, 0):.2f}%)")
    print(f"    Churn     (1) : {counts.get(1, 0):,}  ({pct.get(1, 0):.2f}%)")

    if len(pct) == 2:
        ratio = pct.max() / max(pct.min(), 1e-9)
        print(f"\n  Ratio de déséquilibre : {ratio:.2f}:1")
        if ratio > 3:
            print("  ⚠️  Déséquilibre important → utiliser class_weight='balanced'")
            print("     ou SMOTE / suréchantillonnage si ratio > 5.")

    # Graphique distribution
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].bar(["Non-Churn (0)", "Churn (1)"], counts.values,
                color=["steelblue", "tomato"], edgecolor="white", width=0.5)
    axes[0].set_title("Distribution Churn (effectifs)")
    axes[0].set_ylabel("Nombre de clients")
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + max(counts) * 0.01, f"{v:,}", ha="center", fontsize=10)

    axes[1].pie(counts.values,
                labels=["Non-Churn (0)", "Churn (1)"],
                autopct="%1.1f%%",
                colors=["steelblue", "tomato"],
                startangle=90,
                explode=(0, 0.05))
    axes[1].set_title("Répartition Churn (%)")
    plt.suptitle("Distribution de la variable cible Churn", fontsize=13, y=1.02)
    plt.tight_layout()
    churn_fig_path = figures_dir / "churn_distribution.png"
    fig.savefig(churn_fig_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n  Graphique sauvegardé : {churn_fig_path}")

    # Taux de churn par segment RFM
    for seg_col in ["RFMSegment", "CustomerType", "LoyaltyLevel", "SpendingCat"]:
        if seg_col in df.columns:
            df_temp = df.copy()
            df_temp["_churn"] = y
            churn_by_seg = (
                df_temp.groupby(seg_col)["_churn"]
                .agg(["mean", "count"])
                .rename(columns={"mean": "churn_rate", "count": "n_clients"})
                .sort_values("churn_rate", ascending=False)
            )
            churn_by_seg["churn_rate"] = (churn_by_seg["churn_rate"] * 100).round(2)
            print(f"\n  Taux de churn par {seg_col} (%) :")
            print(churn_by_seg.to_string())

            # Graphique
            fig, ax = plt.subplots(figsize=(8, 4))
            churn_by_seg["churn_rate"].sort_values().plot(kind="barh", ax=ax, color="tomato")
            ax.set_title(f"Taux de churn par {seg_col} (%)")
            ax.set_xlabel("Churn rate (%)")
            plt.tight_layout()
            seg_path = figures_dir / f"churn_by_{seg_col.lower()}.png"
            fig.savefig(seg_path, dpi=150)
            plt.close(fig)
            print(f"  Graphique sauvegardé : {seg_path}")

    return y


# ============================================================
# 6) ACP (PCA) — RÉDUCTION DIMENSIONNELLE
# ============================================================

def analyse_pca(df: pd.DataFrame, y: pd.Series, figures_dir: Path) -> None:
    """
    Applique l'ACP sur les colonnes numériques :
    - Scree plot (variance expliquée par composante)
    - Projection 2D colorée par Churn
    - Table des loadings (contributions des features)
    """
    section("6) ACP — ANALYSE EN COMPOSANTES PRINCIPALES")

    num_df = df.select_dtypes(include="number").copy()

    # Imputation des NaN (nécessaire avant PCA)
    imputer = SimpleImputer(strategy="median")
    data_imp = imputer.fit_transform(num_df)

    # Normalisation (obligatoire pour PCA)
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data_imp)

    # ACP complète pour scree plot
    n_comp_full = min(len(num_df.columns), 20)
    pca_full = PCA(n_components=n_comp_full, random_state=42)
    pca_full.fit(data_scaled)

    ev = pca_full.explained_variance_ratio_
    cumul = np.cumsum(ev)

    # Table variance expliquée
    ev_df = pd.DataFrame({
        "composante":                  [f"PC{i+1}" for i in range(len(ev))],
        "variance_expliquee_pct":       (ev * 100).round(2),
        "variance_cumulee_pct":         (cumul * 100).round(2),
    })
    print("\n  Variance expliquée par composante :")
    print(ev_df.to_string(index=False))

    # Nombre de composantes pour 80% de variance
    n_80 = int(np.searchsorted(cumul, 0.80)) + 1
    print(f"\n  → {n_80} composantes suffisent pour expliquer 80% de la variance.")
    print(f"  → Réduction : {len(num_df.columns)} features → {n_80} composantes")

    # --- Scree plot ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].bar(range(1, len(ev) + 1), ev * 100, color="steelblue", edgecolor="white")
    axes[0].plot(range(1, len(ev) + 1), ev * 100, "o-", color="navy", lw=1.5)
    axes[0].set_title("Scree Plot — Variance expliquée par composante")
    axes[0].set_xlabel("Composante principale")
    axes[0].set_ylabel("Variance expliquée (%)")
    axes[0].axhline(y=5, color="red", linestyle="--", alpha=0.5, label="Seuil 5%")
    axes[0].legend()

    axes[1].plot(range(1, len(cumul) + 1), cumul * 100, "o-", color="green", lw=2)
    axes[1].axhline(y=80, color="orange", linestyle="--", label="80% variance")
    axes[1].axhline(y=95, color="red",    linestyle="--", label="95% variance")
    axes[1].set_title("Variance cumulée")
    axes[1].set_xlabel("Nombre de composantes")
    axes[1].set_ylabel("Variance cumulée (%)")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    scree_path = figures_dir / "pca_scree_plot.png"
    fig.savefig(scree_path, dpi=150)
    plt.close(fig)
    print(f"\n  Scree plot sauvegardé : {scree_path}")

    # --- Projection 2D ---
    pca_2d = PCA(n_components=2, random_state=42)
    X_pca  = pca_2d.fit_transform(data_scaled)

    fig, ax = plt.subplots(figsize=(9, 6))
    if not y.empty and len(y) == len(X_pca):
        scatter = ax.scatter(
            X_pca[:, 0], X_pca[:, 1],
            c=y.values, cmap="RdYlGn_r",
            alpha=0.5, s=15, edgecolors="none",
        )
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label("Churn (0=Non, 1=Oui)")
        # Légende manuelle
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor="green", label="Non-Churn (0)"),
            Patch(facecolor="red",   label="Churn (1)"),
        ]
        ax.legend(handles=legend_elements, loc="upper right")
    else:
        ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.4, s=15)

    pc1_var = pca_2d.explained_variance_ratio_[0] * 100
    pc2_var = pca_2d.explained_variance_ratio_[1] * 100
    ax.set_xlabel(f"PC1 ({pc1_var:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({pc2_var:.1f}% variance)")
    ax.set_title("ACP — Projection 2D (colorée par Churn)")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    pca2d_path = figures_dir / "pca_2d_projection.png"
    fig.savefig(pca2d_path, dpi=150)
    plt.close(fig)
    print(f"  Projection 2D sauvegardée : {pca2d_path}")

    # --- Loadings (Top contributions) ---
    loadings = pd.DataFrame(
        pca_2d.components_.T,
        index=num_df.columns,
        columns=["PC1", "PC2"],
    ).round(4)
    loadings["abs_PC1"] = loadings["PC1"].abs()
    loadings["abs_PC2"] = loadings["PC2"].abs()

    print("\n  Top 10 features contribuant à PC1 :")
    print(loadings.nlargest(10, "abs_PC1")[["PC1"]].to_string())
    print("\n  Top 10 features contribuant à PC2 :")
    print(loadings.nlargest(10, "abs_PC2")[["PC2"]].to_string())

    # Sauvegarde loadings
    loadings_clean = loadings.drop(columns=["abs_PC1", "abs_PC2"])
    loadings_clean.to_csv(REPORTS_DIR / "pca_loadings.csv")
    print(f"\n  Loadings sauvegardés : {REPORTS_DIR / 'pca_loadings.csv'}")

    # Sauvegarde variance expliquée
    ev_df.to_csv(REPORTS_DIR / "pca_explained_variance.csv", index=False)
    print(f"  Variance expliquée sauvegardée : {REPORTS_DIR / 'pca_explained_variance.csv'}")


# ============================================================
# 7) FEATURE ENGINEERING
# ============================================================

def feature_engineering_preview(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crée et affiche des nouvelles features dérivées comme demandé dans l'énoncé.
    Ne modifie pas le dataset brut — retourne un DataFrame enrichi pour aperçu.
    """
    section("7) FEATURE ENGINEERING — NOUVELLES VARIABLES")

    df_fe = df.copy()
    created = []

    # --- Features RFM dérivées ---
    if "MonetaryTotal" in df_fe.columns and "Recency" in df_fe.columns:
        df_fe["MonetaryPerDay"] = (
            safe_to_numeric(df_fe["MonetaryTotal"]) /
            (safe_to_numeric(df_fe["Recency"]) + 1)
        )
        created.append("MonetaryPerDay  = MonetaryTotal / (Recency + 1)")

    if "MonetaryTotal" in df_fe.columns and "Frequency" in df_fe.columns:
        freq = safe_to_numeric(df_fe["Frequency"]).replace(0, np.nan)
        df_fe["AvgBasketValue"] = safe_to_numeric(df_fe["MonetaryTotal"]) / freq
        created.append("AvgBasketValue  = MonetaryTotal / Frequency")

    if "CustomerTenure" in df_fe.columns and "Recency" in df_fe.columns:
        df_fe["TenureRatio"] = (
            safe_to_numeric(df_fe["CustomerTenure"]) /
            (safe_to_numeric(df_fe["Recency"]) + 1)
        )
        created.append("TenureRatio     = CustomerTenure / (Recency + 1)")

    if "Frequency" in df_fe.columns and "CustomerTenure" in df_fe.columns:
        tenure = safe_to_numeric(df_fe["CustomerTenure"]).replace(0, np.nan)
        df_fe["FrequencyIntensity"] = safe_to_numeric(df_fe["Frequency"]) / tenure
        created.append("FrequencyIntensity = Frequency / CustomerTenure")

    # --- Feature depuis LastLoginIP ---
    if "LastLoginIP" in df_fe.columns:
        def is_private(ip):
            try:
                parts = str(ip).split(".")
                if len(parts) != 4:
                    return 0
                o1, o2 = int(parts[0]), int(parts[1])
                if o1 == 10 or (o1 == 172 and 16 <= o2 <= 31) or (o1 == 192 and o2 == 168):
                    return 1
            except Exception:
                pass
            return 0

        df_fe["IP_IsPrivate"] = df_fe["LastLoginIP"].apply(is_private)
        created.append("IP_IsPrivate    = 1 si IP privée (10.x / 172.16-31.x / 192.168.x)")

    # --- Feature depuis RegistrationDate ---
    date_candidates = [
        c for c in df_fe.columns
        if c.lower() in {"registdate", "registrationdate", "registerdate", "signupdate"}
    ]
    if date_candidates:
        dcol = date_candidates[0]
        dt = safe_to_datetime(df_fe[dcol])
        df_fe["RegistrationYear"]    = dt.dt.year
        df_fe["RegistrationMonth"]   = dt.dt.month
        df_fe["RegistrationWeekday"] = dt.dt.weekday
        max_dt = dt.max()
        if pd.notna(max_dt):
            df_fe["CustomerTenureDays"] = (max_dt - dt).dt.days
        created.extend([
            f"RegistrationYear    (depuis {dcol})",
            f"RegistrationMonth   (depuis {dcol})",
            f"RegistrationWeekday (depuis {dcol})",
            f"CustomerTenureDays  = ancienneté en jours",
        ])

    print(f"\n  {len(created)} nouvelle(s) feature(s) créée(s) :")
    for f in created:
        print(f"    + {f}")

    # Aperçu des nouvelles colonnes
    new_cols = [c for c in df_fe.columns if c not in df.columns]
    if new_cols:
        print(f"\n  Aperçu des nouvelles features :")
        print(df_fe[new_cols].describe().round(2).to_string())

    return df_fe


# ============================================================
# 8) EXPORT DES RAPPORTS
# ============================================================

def export_rapports(
    df: pd.DataFrame,
    outlier_df: pd.DataFrame,
    domain_df: pd.DataFrame,
    high_corr: pd.DataFrame,
    reports_dir: Path,
) -> None:
    """Sauvegarde tous les rapports CSV dans reports/."""
    section("8) EXPORT DES RAPPORTS")

    # Rapport colonnes
    rows = []
    for c in df.columns:
        rows.append({
            "column":        c,
            "dtype":         str(df[c].dtype),
            "n_unique":      int(df[c].nunique(dropna=True)),
            "missing_count": int(df[c].isna().sum()),
            "missing_pct":   round(float(df[c].isna().mean() * 100), 2),
            "sample_values": ", ".join(map(str, df[c].dropna().astype(str).head(3).tolist())),
        })
    col_report = (
        pd.DataFrame(rows)
        .sort_values(["missing_pct", "n_unique"], ascending=[False, False])
    )

    col_report.to_csv(reports_dir / "exploration_columns_report.csv", index=False)
    outlier_df.to_csv(reports_dir / "exploration_outliers_report.csv", index=False)

    if not domain_df.empty:
        domain_df.to_csv(reports_dir / "exploration_domain_checks.csv", index=False)

    if not high_corr.empty:
        high_corr.to_csv(reports_dir / "exploration_high_correlation.csv", index=False)

    print(f"\n  Rapports sauvegardés dans : {reports_dir}")
    print("    ✅ exploration_columns_report.csv")
    print("    ✅ exploration_outliers_report.csv")
    if not domain_df.empty:
        print("    ✅ exploration_domain_checks.csv")
    if not high_corr.empty:
        print("    ✅ exploration_high_correlation.csv")
    print(f"\n  Figures sauvegardées dans : {reports_dir / 'figures'}")


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    print("\n" + "=" * 60)
    print("  EXPLORATION — Retail Customers Churn Dataset")
    print("=" * 60)

    # Vérification existence dataset
    if not RAW_PATH.exists():
        print(f"\n❌ Dataset introuvable : {RAW_PATH}")
        print("   → Place le fichier CSV dans data/raw/")
        sys.exit(1)

    # Chargement
    df = pd.read_csv(RAW_PATH)
    df.columns = [str(c).strip() for c in df.columns]
    print(f"\n✅ Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

    # ── Sections ──────────────────────────────────────────────
    num_cols, cat_cols = analyse_structure(df)
    outlier_df         = analyse_qualite(df, num_cols, cat_cols)
    domain_df          = controles_metier(df)
    high_corr          = analyse_correlation(df, num_cols, FIGURES_DIR)
    y                  = analyse_churn(df, FIGURES_DIR)
    analyse_pca(df, y, FIGURES_DIR)
    feature_engineering_preview(df)
    export_rapports(df, outlier_df, domain_df, high_corr, REPORTS_DIR)

    print("\n" + "=" * 60)
    print("  ✅ Exploration terminée avec succès !")
    print(f"  Rapports : {REPORTS_DIR}")
    print(f"  Figures  : {FIGURES_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    main()

PROJECT_ROOT : c:\Users\User\Desktop\projet_ml_retail
RAW_PATH     : c:\Users\User\Desktop\projet_ml_retail\data\raw\retail_customers_COMPLETE_CATEGORICAL.csv
RAW exists?  : True

  EXPLORATION — Retail Customers Churn Dataset

✅ Dataset chargé : 4,372 lignes × 52 colonnes

  1) STRUCTURE GLOBALE
Dimensions : 4,372 lignes × 52 colonnes

Premières lignes :
   CustomerID  Recency  Frequency  MonetaryTotal  MonetaryAvg  MonetaryStd  MonetaryMin  MonetaryMax  TotalQuantity  AvgQuantityPerTransaction  MinQuantity  MaxQuantity  CustomerTenureDays  FirstPurchaseDaysAgo  PreferredDayOfWeek  PreferredHour  PreferredMonth  WeekendPurchaseRatio  AvgDaysBetweenPurchases  UniqueProducts  UniqueDescriptions  AvgProductsPerTransaction  UniqueCountries  NegativeQuantityCount  ZeroPriceCount  CancelledTransactions  ReturnRatio  TotalTransactions  UniqueInvoices  AvgLinesPerInvoice   Age RegistrationDate NewsletterSubscribed     LastLoginIP  SupportTicketsCount  SatisfactionScore RFMSegment AgeCategory 

C:\Users\User\AppData\Local\Temp\ipykernel_30844\3812176534.py:61: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, errors="coerce", dayfirst=True)


  Heatmap sauvegardée : c:\Users\User\Desktop\projet_ml_retail\reports\figures\correlation_heatmap.png

  Paires avec |corrélation| ≥ 0.8 :
                feature_1             feature_2  correlation
                Frequency        UniqueInvoices     1.000000
    NegativeQuantityCount CancelledTransactions     1.000000
           UniqueProducts    UniqueDescriptions     0.999930
              MonetaryMin           MonetaryMax    -0.993947
              MonetaryStd           MinQuantity    -0.974047
              MonetaryStd           MaxQuantity     0.972585
              MonetaryStd           MonetaryMin    -0.967325
              MonetaryStd           MonetaryMax     0.966135
AvgProductsPerTransaction    AvgLinesPerInvoice     0.963210
              MinQuantity           MaxQuantity    -0.961174
            MonetaryTotal         TotalQuantity     0.921649
              MonetaryMax           MaxQuantity     0.920975
              MonetaryMin           MinQuantity     0.919252
      

C:\Users\User\AppData\Local\Temp\ipykernel_30844\3812176534.py:61: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, errors="coerce", dayfirst=True)
